In [1]:
# ================================================================================================================
# 1. IMPORTACION DE LIBRERIAS
# ================================================================================================================
import pymupdf
import pandas as pd
import os
import re
from datetime import datetime
import unicodedata

ruta1 = r"D:\Archivos_Python\Canal2\Factura EJESA\T2.pdf"
ruta2 = r"D:\Archivos_Python\Canal2\Factura EJESA\T3.pdf"
carpeta = r"D:\Archivos_Python\Canal2\Factura EJESA"

area_1 = {"x1": 11.57,"x2":446.00,"y1": 11.85,"y2":175.56,"width":434.52,"height":163.71}
area_2 = {"x1":449.15,"x2":597.56,"y1": 11.85,"y2":177.09,"width":148.41,"height":165.24}
area_3 = {"x1": 13.10,"x2":201.30,"y1":249.77,"y2":343.10,"width":188.19,"height": 93.33}
area_4 = {"x1":205.88,"x2":603.68,"y1":179.39,"y2":349.22,"width":397.80,"height":169.83}
area_5 = {"x1":306.10,"x2":599.86,"y1":352.28,"y2":548.12,"width":293.76,"height":195.84}
area_6 = {"x1":199.76,"x2":596.80,"y1":564.18,"y2":650.63,"width":397.035,"height":86.44}
area_7 = {"x1":306.10,"x2":601.39,"y1":652.92,"y2":830.40,"width":295.29,"height":177.48}

rect_1 = pymupdf.Rect(area_1["x1"], area_1["y1"],area_1["x2"],area_1["y2"])
rect_2 = pymupdf.Rect(area_2["x1"], area_2["y1"],area_2["x2"],area_2["y2"])
rect_3 = pymupdf.Rect(area_3["x1"], area_3["y1"],area_3["x2"],area_3["y2"])
rect_4 = pymupdf.Rect(area_4["x1"], area_4["y1"],area_4["x2"],area_4["y2"])
rect_5 = pymupdf.Rect(area_5["x1"], area_5["y1"],area_5["x2"],area_5["y2"])
rect_6 = pymupdf.Rect(area_6["x1"], area_6["y1"],area_6["x2"],area_6["y2"])
rect_7 = pymupdf.Rect(area_7["x1"], area_7["y1"],area_7["x2"],area_7["y2"])

# ================================================================================================================
# 2. DEFINICION DE FUNCIONES A UTILIZAR
# ================================================================================================================
# Función para quitar tildes/acentos
def quitar_tildes(texto: str) -> str:
    if not isinstance(texto, str):
        return texto
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def buscar_texto(lista: list[str], TextoBuscado, Posicion) -> str | None:
    for i, texto in enumerate(lista):
        if TextoBuscado.upper() in texto.upper():  # uso upper() para evitar problemas de mayúsculas/minúsculas
            if i + 1 < len(lista):      # verifico que exista un elemento siguiente
                return lista[i+Posicion].strip()
    return None
def buscar_patron(lista: list[str],patron) -> str | None:
    for linea in lista:
        match = re.search(patron, linea)
        if match:
            return match.group()
    return None
def encontrar_concepto(Lista, TextoBuscar, PosicionTexto):
    concepto = [ x for x in Lista if TextoBuscar in x ][0].split(" ")[PosicionTexto].strip()
    return concepto

def obtener_precio(row):
    texto = row[1]
    if "$/kWh" in texto:
        match = re.search(r"(\d+(?:\.\d+)?)\s*\$/kWh", texto)
        if match:
            return match.group(1)
        else:
            return row[2]
    else:
        return row[2]

In [2]:
def PROCESAR_pdf(ruta):
    doc = pymupdf.open(ruta)
    pagina = doc[0]

    # Extraer texto del área
    cuadro_1 = pagina.get_text("text", clip=rect_1)
    cuadro_2 = pagina.get_text("text", clip=rect_2)
    cuadro_3 = pagina.get_text("words", clip=rect_3)
    cuadro_4 = pagina.get_text("words", clip=rect_4)
    cuadro_5 = pagina.get_text("words", clip=rect_5)

    Lista_box1 = cuadro_1.split("\n")
    Lista_box2 = cuadro_2.split("\n")

    Lista_resumen = [ (round(tupla[3]/10), 1, tupla[4] ) for tupla in cuadro_3]  #round(tupla[3]*1.7/100)
    df = pd.DataFrame(Lista_resumen, columns=['Fila', 'Col', 'Valor'])
    df = df.pivot_table(index='Fila', columns='Col', values='Valor', aggfunc=lambda x: ' '.join(str(v) for v in x))
    Lista_box3 = df[1].tolist()

    # Datos del proveedor
    titular = buscar_texto(Lista_box1,"TITULAR",1)
    DNIcuit = buscar_texto(Lista_box1, "CUIT",0).split("CUIT")[1].split("ACTIVIDAD")[0].strip()
    nro_factura = buscar_patron(Lista_box1, r"\d{2}-\d{8}" )
    periodo = buscar_patron(Lista_box2, r"\d{2}/\d{4}")
    servicio = buscar_texto(Lista_box2,"SERVICIO",1)
    vencimiento = buscar_texto(Lista_box2,"VENCIMIENTO",1)
    total_ejesa = buscar_texto(Lista_box2,"EJESA",1)
    total_agua  = buscar_texto(Lista_box2,"APYSJ",1)
    total_pagar = buscar_texto(Lista_box2,"PAGAR",1)

    # Datos de la factura medicion
    fecha_anterior = buscar_texto(Lista_box3,"ANTERIOR",0).split(" ")[1]
    fecha_actual = buscar_texto(Lista_box3,"ACTUAL",0).split(" ")[1]
    tipo_tarifa = buscar_texto(Lista_box3,"TARIFA",1).split(" ")[-1]
    # Calcular diferencia
    fmt = "%d/%m/%Y"
    f1 = datetime.strptime(fecha_anterior, fmt)
    f2 = datetime.strptime(fecha_actual, fmt)
    dias_medicion_ejesa = (f2 - f1).days

    Lista_medicion = [ (int(tupla[3]/10), 1, tupla[4] ) for tupla in cuadro_4]
    df_2 = pd.DataFrame(Lista_medicion, columns=['Fila', 'Col', 'Valor'])
    df_2 = df_2.pivot_table(index='Fila', columns='Col', values='Valor', aggfunc=lambda x: ' '.join(str(v) for v in x))
    Lista_box4 = df_2[1].tolist()
    consumo = buscar_texto(Lista_box4,"TOTAL",0).split(" ")[-2:][0]
    consumo_facturado = consumo
    L_fijo = buscar_texto(Lista_box4,"CARGO FIJO",0).split(" ")[-1]
    L_uso = buscar_texto(Lista_box4,"CARGO POR USO",0).split(" ")[-4:]
    max_uso = str(max([float(L_uso[0].replace(',', '.')),float(L_uso[1].replace(',', '.'))])).replace(".",",")
    Lista_datos = [
        ["BASICO", "Cargo Fijo", None, None, 1, L_fijo, L_fijo],
        ["BASICO", "Cargo por Uso de Red", None, None, max_uso] + L_uso[-2:],
        ["BASICO", "Horas Puntas"] + buscar_texto(Lista_box4,"PUNTA",0).split(" ")[-5:],
        ["BASICO", "Horas Valle Nocturno"] + buscar_texto(Lista_box4,"VALLE",0).split(" ")[-5:],
        ["BASICO", "Horas Restantes"] + buscar_texto(Lista_box4,"RESTANTES",0).split(" ")[-5:]
    ]
    df_liq_1 = pd.DataFrame(Lista_datos, columns=["Grupo","Concepto","medicion_anterior","medicion_actual","Cant","Precio_Unit", "Importe"])

    # Determinar el df_liq: es el detalle de la factura
    Lista_box5 = [ ( int(tupla[1]), 1 if tupla[2] < 500 else 2, tupla[4] ) for tupla in cuadro_5]
    tb1 = pd.DataFrame(Lista_box5, columns=['Fila', 'Col', 'Valor'])
    tb2 = tb1.pivot_table(index='Fila', columns='Col', values='Valor', aggfunc=lambda x: ' '.join(str(v) for v in x)).reset_index()
    tb2[1] = tb2[1].apply(quitar_tildes)
    tb2["Grupo"] = tb2[1].apply(
        lambda x: (
            "REDONDEO" if isinstance(x, str) and "AJUSTE POR REDONDEO" in x.upper()
            else x if isinstance(x, str) and x.upper() == "ENERGIA ELECTRICA" and "SUBTOTAL" not in x #else x if isinstance(x, str) and x.upper() == x and "SUBTOTAL" not in x
            else x if isinstance(x, str) and x.upper() == "TASAS E IMPUESTOS" and "SUBTOTAL" not in x
            else x if isinstance(x, str) and x.upper() == "OTROS CARGOS" and "SUBTOTAL" not in x
            else None
        )
    )
    tb2["Grupo"] = tb2["Grupo"].ffill()
    tb_filtrado = tb2[(~tb2[1].str.contains("TOTAL|ENERGIA|OTROS|TASAS|Basico", case=True, na=False))]
    tb_filtrado["Precio_Unit"] = tb_filtrado.apply(obtener_precio, axis=1)
    tb_filtrado["Cant"] = tb_filtrado[1].apply(lambda x: consumo_facturado if "$/kWh" in x else 1)
    df_liq_2 = tb_filtrado[["Grupo",1, 2,"Cant","Precio_Unit"]]
    df_liq_2 = df_liq_2.rename(columns={1: "Concepto", 2: "Importe"})
    df_liq_ejesa = pd.concat([df_liq_1, df_liq_2], ignore_index=True)
    df_liq_ejesa["Emp_serv"]="EJESA"
    df_liq_ejesa["fecha_anterior"] = fecha_anterior
    df_liq_ejesa["fecha_actual"] = fecha_actual
    df_liq_ejesa["consumo"] = consumo
    df_liq_ejesa["tipo_tarifa"] = tipo_tarifa
    df_liq_ejesa["consumo_facturado"] = consumo_facturado
    df_liq_ejesa["dias_medicion"] = dias_medicion_ejesa

    if total_agua == "0,00":
        df_final = df_liq_ejesa
    else:
        cuadro_6 = pagina.get_text("words", clip=rect_6)
        cuadro_7 = pagina.get_text("words", clip=rect_7)
        Lista_6 = [ (int(int(tupla[3])/10), 1, tupla[4] ) for tupla in cuadro_6]
        df_ag = pd.DataFrame(Lista_6, columns=['Fila', 'Col', 'Valor'])
        df_ag = df_ag.pivot_table(index='Fila', columns='Col', values='Valor', aggfunc=lambda x: ' '.join(str(v) for v in x))
        Lista_box6 = df_ag[1].tolist()
        fecha_ant_agua = encontrar_concepto(Lista_box6,"ANTERIOR",1)
        fecha_act_agua = encontrar_concepto(Lista_box6,"ACTUAL",1)
        med_ant_agua = encontrar_concepto(Lista_box6,"ANTERIOR",2)
        med_act_agua = encontrar_concepto(Lista_box6,"ACTUAL",2)
        fecha_ant_agua = encontrar_concepto(Lista_box6,"ANTERIOR",1)
        fecha_ant_agua = encontrar_concepto(Lista_box6,"ANTERIOR",1)
        dias_medicion_agua = encontrar_concepto(Lista_box6,"DIFERENCIA",1)
        consumo_agua = encontrar_concepto(Lista_box6,"DIFERENCIA",2)
        uso_parametro = encontrar_concepto(Lista_box6,"USO",1)
        consumo_facturado_agua = encontrar_concepto(Lista_box6,"DIFERENCIA",-1)
        
        Lista_box7 = [ ( int(tupla[3]), 1 if tupla[2] < 500 else 2, tupla[4] ) for tupla in cuadro_7]
        tb_ag = pd.DataFrame(Lista_box7, columns=['Fila', 'Col', 'Valor'])
        tb_ag = tb_ag.pivot_table(index='Fila', columns='Col', values='Valor', aggfunc=lambda x: ' '.join(str(v) for v in x)).reset_index()
        tb_ag[1] = tb_ag[1].apply(quitar_tildes)
        tb_ag["Grupo"] = tb_ag[1].apply(
            lambda x: (
                    x if isinstance(x, str) and x.upper() == "SERVICIOS SANITARIOS" and "SUBTOTAL" not in x #else x if isinstance(x, str) and x.upper() == x and "SUBTOTAL" not in x
                else x if isinstance(x, str) and x.upper() == "TASAS E IMPUESTOS" and "SUBTOTAL" not in x
                else x if isinstance(x, str) and x.upper() == "OTROS CONCEPTOS" and "SUBTOTAL" not in x
                else None
            )
        )
        tb_ag["Grupo"] = tb_ag["Grupo"].ffill()

        tb_filtrado_ag = tb_ag[(~tb_ag[1].str.contains("TOTAL|SERVICIOS|OTROS|TASAS", case=True, na=False)) ]
        df_liq_agua = tb_filtrado_ag[["Grupo",1, 2]]
        df_liq_agua.columns = ["Grupo", "Concepto", "Importe"]
        df_liq_agua["Emp_serv"]="AGUA"
        df_liq_agua["fecha_anterior"] = fecha_ant_agua
        df_liq_agua["fecha_actual"] = fecha_act_agua
        df_liq_agua["dias_medicion"] = dias_medicion_agua
        df_liq_agua["medicion_anterior"] = med_ant_agua
        df_liq_agua["medicion_actual"] = med_act_agua
        df_liq_agua["consumo"] = consumo_agua
        df_liq_agua["tipo_tarifa"] = uso_parametro
        df_liq_agua["consumo_facturado"] = consumo_facturado_agua
        df_liq_agua["Precio_Unit"] = df_liq_agua["Importe"].copy()
        df_liq_agua["Cant"] = 1
        df_final = pd.concat([df_liq_ejesa,df_liq_agua])
    df_final

    # Crear el diccionario con los datos extraídos
    factura_dict = {
        "titular": titular,
        "DNIcuit": DNIcuit,
        "nro_factura": nro_factura,
        "periodo": periodo,
        "servicio": servicio,
        "vencimiento": vencimiento,
        "total_ejesa": total_ejesa,
        "total_agua": total_agua,
        "total_pagar": total_pagar,
        "detalle_liquidacion": df_final.to_dict(orient="records")  # convierte el DataFrame en lista de diccionarios
    }
    return factura_dict

In [3]:
resultados = []
for archivo in os.listdir(carpeta):
    if archivo.endswith(".pdf"):
        ruta = os.path.join(carpeta, archivo)
        print(archivo)
        factura = PROCESAR_pdf(ruta)
        resultados.append(factura)
df_final = pd.DataFrame(resultados)

T2.pdf
T3.pdf


In [ ]:
# 1. Explode: cada diccionario se convierte en una fila
df_exploded = df_final.explode("detalle_liquidacion")
# 2. Normalizar: convertir los diccionarios en columnas
df_detalle = pd.json_normalize(df_exploded["detalle_liquidacion"])
# 3. Combinar con las demás columnas
df_facturas = pd.concat([df_exploded.drop(columns=["detalle_liquidacion"]), df_detalle], axis=1)
df_facturas

In [ ]:
df_facturas = df_facturas[["titular","DNIcuit","nro_factura","periodo","servicio","vencimiento","total_ejesa","total_agua",
              "total_pagar","Emp_serv","fecha_anterior","fecha_actual","dias_medicion","medicion_anterior","medicion_actual",
              "consumo","tipo_tarifa","consumo_facturado","Grupo","Concepto","Importe","Precio_Unit","Cant"]]